# Topics in Quantitative Finance - Homework 3 Solution

Assigned: Friday, July 31, 2026.
Due: **Monday, August 3, 2026** by 1PM. 

Late homework **will not be accepted**.

$$
\newcommand{\supp}{\mathrm{supp}}
\newcommand{\E}{\mathbb{E} }
\newcommand{\Eof}[1]{\mathbb{E}\left[ #1 \right]}
\def\Cov{{ \mbox{Cov} }}
\def\Var{{ \mbox{Var} }}
\newcommand{\1}{\mathbf{1} }
\newcommand{\PP}{\mathbb{P} }
\newcommand{\Pof}[1]{\mathbb{P}\left[ #1 \right]}
%\newcommand{\Pr}{\mathrm{Pr} }
\newcommand{\QQ}{\mathbb{Q} }
\newcommand{\RR}{\mathbb{R} }
\newcommand{\DD}{\mathbb{D} }
\newcommand{\HH}{\mathbb{H} }
\newcommand{\spn}{\mathrm{span} }
\newcommand{\cov}{\mathrm{cov} }
\newcommand{\sgn}{\mathrm{sgn} }
\newcommand{\HS}{\mathcal{L}_{\mathrm{HS}} }
%\newcommand{\HS}{\mathrm{HS} }
\newcommand{\trace}{\mathrm{trace} }
\newcommand{\LL}{\mathcal{L} }
%\newcommand{\LL}{\mathrm{L} }
\newcommand{\s}{\mathcal{S} }
\newcommand{\ee}{\mathcal{E} }
\newcommand{\ff}{\mathcal{F} }
\newcommand{\hh}{\mathcal{H} }
\newcommand{\bb}{\mathcal{B} }
\newcommand{\dd}{\mathcal{D} }
\newcommand{\g}{\mathcal{G} }
\newcommand{\p}{\partial}
\newcommand{\half}{\frac{1}{2} }
\newcommand{\T}{\mathcal{T} }
\newcommand{\bi}{\begin{itemize}}
\newcommand{\ei}{\end{itemize}}
\newcommand{\beq}{\begin{equation}}
\newcommand{\eeq}{\end{equation}}
\newcommand{\beas}{\begin{align*}}
\newcommand{\eeas}{\end{align*}}
\newcommand{\cO}{\mathcal{O}}
\newcommand{\cF}{\mathcal{F}}
\newcommand{\cL}{\mathcal{L}}
\newcommand{\BS}{\text{BS}}
$$

<font color = "red">Homework is to be done by each student individually.  To receive full credit, you must email a completed copy of this Jupyter notebook to TAs at [topics_in_qf@163.com](mailto:topics_in_qf@163.com) by the due date and time.  All codes must run correctly and solutions must be written up neatly in Markdown/LaTeX format. If you encounter problems with Jupyter notebook, please contact TA [李新宇](mailto:xinyu911@stu.pku.edu.cn) or [林文鑫](mailto:vincent_lin@stu.pku.edu.cn).

## Name: <font color=blue>龚天翔</font>

## Delta and delta-gamma hedging

### 1. (25 points)

A portfolio consists of 
- a long position in 500 shares of a nondividend paying stock with spot price $\$20$,
- a short position in 1000 puts struck at $\$25$ and expiring in 3 months on the stock, assumed lognormally distributed with $30\%$ volatility, 
- $\$10,000$ in money account with annual interest rate $4\%$ continuously compounding.

Answer the following questions.

* (a) What is the value of the portfolio?
* (b) How do you adjust the holdings of stock shares and cash amounts in the portfolio in order to make it delta neutral without changing the postion in puts?
* (c) How do you adjust the portfolio in order to make it delta-gamma neutral by adding position in calls struck at $\$30$? Position in puts cannot be altered. 
* (d) A month later the stock goes up to $\$24$. Determine the value of the delta-neuralized portfolio in (b). 
* (e) How do you rebalance the portfolio in (d) so it remains delta neutral?    


You may consider using the code provided in the cell below for the calculation of deltas and gammas of call and put. 

In [9]:
# as always, import required modules and functions
import numpy as np
from numpy import sqrt, log, exp
import matplotlib.pyplot as plt
import scipy.stats as ss
from scipy.stats import norm
import seaborn as sns

In [10]:
# Black-Scholes formulas
# call
def bs_call(s, K, sigma, t, r=0, d=0):
    d1 = (log(s/K) + (r - d)*t)/(sigma*sqrt(t)) + sigma*sqrt(t)/2
    d2 = d1 - sigma*sqrt(t)
    
    c = s*exp(-d*t)*norm.cdf(d1) - K*exp(-r*t)*norm.cdf(d2)
    delta = exp(-d*t)*norm.cdf(d1)
    gamma = norm.pdf(d1)/s/sigma/sqrt(t)
    
    return {'c': c, 'delta': delta, 'gamma': gamma}

#put
def bs_put(s, K, sigma, t, r=0, d=0):
    d1 = (log(s/K) + (r - d)*t)/(sigma*sqrt(t)) + sigma*sqrt(t)/2
    d2 = d1 - sigma*sqrt(t)
    
    p = K*exp(-r*t)*norm.cdf(-d2) - s*exp(-d*t)*norm.cdf(-d1)
    delta = -exp(-d*t)*norm.cdf(-d1)
    gamma = norm.pdf(d1)/s/sigma/sqrt(t)
    
    return {'p': p, 'delta': delta, 'gamma': gamma}

## <font color=blue> Solution 1. </font>

# (1a)
已知条件：(以下涉及价格与现金的单位,为便于书写而省去"$"；涉及时间的单位均为“年”,为便于书写而省去)  \
股票现价：$S_0=20$ \
股票数量：$N_S=500$ \
put option的strike price：$K_P=25$ \
put option底层stock的volatility：$\sigma=0.3$ \
其股息率：$d=0$ \
put option的time to expiry：$\tau=0.25$ \
put option的数量：$N_P = -1000$ （由于是卖出，故为负值） \
现金：$Cash_0=10000$ \
利率：$r=0.04$ 

利用 Black-Scholes 公式计算put option的价格
$$
P_0 = K_P e^{-r\tau} N(-d_2) - S_0 e^{-d\tau} N(-d_1)
$$
其中  
$$
d_1 = \frac{\log\left(\frac{S_0e^{-d\tau}}{K_Pe^{-r\tau}}\right)}{\sigma\sqrt\tau}+ \frac{\sigma\sqrt\tau}2, \qquad d_2 = d_1 - \sigma \sqrt\tau
$$
代入已知数据可得：
$$
P_0 \approx 4.8678
$$
故此portfolio的value为：
$$
V_0 = N_S \cdot S_0 + Cash_0 + N_P \cdot P_0  \approx \$15132.15
$$
具体的计算代码如下：

In [11]:
# (1a) 

# Parameters
S0 = 20
K_P = 25
T = 0.25
r = 0.04
sigma = 0.30
N_S = 500
N_P = -1000
Cash = 10000

# Compute put price and Greeks
put = bs_put(S0, K_P, sigma, T, r)
P0 = put['p']
delta_P0 = put['delta']
gamma_P0 = put['gamma']
# Portfolio value
V0 = N_S * S0 + Cash + N_P * P0

print(f"(a) Initial Portfolio Value: ${V0:.2f}")

(a) Initial Portfolio Value: $15132.15


# (1b)
portfolio的Delta：
$$
\Delta_{\Pi_0}=\frac{\partial V_0}{\partial S_0}=N_S+N_P \cdot \Delta_{P_0}
$$
利用 Black-Scholes 公式计算put option的$\Delta$：
$$
\Delta_{P_0}=\frac{\partial P_0}{\partial S_0}=-N(-d_1) \approx -0.9108
$$
在不改变$N_P$的情况下使$\Delta_{\Pi_0}=0$（delta neutral），可得调整后的新的持有股票数：
$$
N_{S_{dn}}= -N_P \cdot \Delta_{P_0} \approx -910.84
$$
即调整时需要卖出的股票数量为：
$$
N_S-N_{S_{dn}} \approx 1410.84
$$
调整后的新的持有现金数：
$$
Cash_{dn}=Cash_0-(N_{S_{dn}}-N_S)\cdot S_0 \approx \$38216.84
$$
具体的计算代码如下：

In [12]:
# (1b)

# Delta neutral stock position
N_S_dn = -N_P * delta_P0
cash_dn = Cash + (N_S - N_S_dn) * S0

print(f"(b) New Stock Position: {N_S_dn:.2f} shares")
print(f"    New Cash Balance: ${cash_dn:.2f}")

(b) New Stock Position: -910.84 shares
    New Cash Balance: $38216.84


# （1c）
已知call option的strike price：$K_C=30$ \
设持有call option的数量为：$N_C$ \
利用 Black-Scholes 公式计算put option的$\Gamma$:
$$
\Gamma_{P_0}=\frac{\p^2 P_0}{\p S_0^2} = \frac{n(d_1)}{S_0\sigma\sqrt \tau} \approx 0.05375
$$
同理可以利用 Black-Scholes 公式计算call option的价格，$\Delta$和$\Gamma$:
$$
C_0 \approx 0.004751
$$
$$
\Delta_{C_0} \approx 0.005212
$$
$$
\Gamma_{C_0} \approx 0.005001
$$
此时portfolio的value为：
$$
V_n = N_S \cdot S_0 + Cash_0 + N_P \cdot P_0 + N_C \cdot C_0
$$
故其$\Delta$和$\Gamma$为:
$$
\Delta_{\Pi_n}=\frac{\p V_n}{\p S_0}=N_S+N_P \cdot \Delta_{P_0}+N_C \cdot \Delta_{C_0}
$$
$$
\Gamma_{\Pi_n}=\frac{\p^2 V_n}{\p S_0^2}=N_P \cdot \Gamma_{P_0}+N_C \cdot \Gamma_{C_0}
$$
联立上式，在不改变$N_P$的情况下使$\Delta_{\Pi_n}=\Gamma_{\Pi_n}=0$（delta-gamma neutral），可得此时持有的call option数量（即需要买入的call option数量）：
$$
N_{C_{dgn}} = -\frac{N_P \cdot \Gamma_{P_0}}{\Gamma_{C_0}} \approx 10747
$$
调整后持有的股票数：
$$
N_{S_{dgn}} = - (N_P \cdot \Delta_{P_0}+N_{C_{dgn}} \cdot \Delta_{C_0}) \approx -966.86
$$
即调整时需要卖出的股票数量为：
$$
N_S-N_{S_{dgn}}=1466.86
$$
调整后持有的现金数
$$
Cash_{dgn}=Cash_0-(N_{S_{dgn}}-N_S)\cdot S_0 - N_{C_{dgn}}\cdot C_0 \approx \$39286.10
$$
具体计算代码如下：

In [13]:
# (1c)

# Compute call Greeks (K=30)
call = bs_call(S0, 30, sigma, T, r)
delta_C0 = call['delta']
gamma_C0 = call['gamma']
C0 = call['c']

# Gamma neutral
N_C = -(N_P * gamma_P0) / gamma_C0

# Recompute total delta with N_C, then adjust stock
N_S_dg = - (N_P * delta_P0 + N_C * delta_C0)

# Cash adjustment: buy calls, adjust stock
cash_dg = cash_dn - N_C * C0 + (N_S_dn - N_S_dg) * S0

print(f"(c) Number of Calls to Buy: {N_C:.0f}")
print(f"    New Stock Position: {N_S_dg:.2f} shares")
print(f"    New Cash Balance: ${cash_dg:.2f}")

(c) Number of Calls to Buy: 10747
    New Stock Position: -966.86 shares
    New Cash Balance: $39286.10


# (1d)
一个月后， \
股价变为：$S_{new}=24$
option的time to expiry变为：$\tau_{new}=\frac{1}{4}-\frac{1}{12}=\frac{1}{6}$
代入 Black-Scholes 公式可计算出此时put option的价格：
$$
P_{new} \approx 1.6552
$$
而(b)小问中此时portfolio的组成为:
$$
N_P = -1000
$$
$$
N_{S_{dn}} \approx -910.84
$$
$$
Cash_{dn} \approx \$38216.84
$$
其中现金按照利率r增长：
$$
Cash_{new}=Cash_{dn}\cdot e^{\frac{r}{12}} \approx 38344.44
$$
可以计算出此时portfolio的value为：
$$
V_{new}=Cash_{new}+N_P \cdot P_{new}+N_{S_{dn}} \cdot S_{new} \approx \$14829.02
$$
计算代码如下：

In [14]:
# (1d)

# New time to maturity
t_new = T - 1/12

# New put price
put_new = bs_put(24, K_P, sigma, t_new, r)
P_new = put_new['p']
delta_P_new = put_new['delta']

# Cash growth
cash_new_d = cash_dn * exp(r * 1/12)

# Portfolio value
V_d = N_S_dn * 24 + cash_new_d + N_P * P_new

print(f"(d) Portfolio Value = ${V_d:.2f}")

(d) Portfolio Value = $14829.02


# (1e)
利用 Black-Scholes 公式可计算出此时put option的$\Delta$:
$$
\Delta_{P_{new}} \approx -0.5861
$$
在不改变$N_P$的情况下，使此时portfolio的$\Delta$为0（delta neutral）:
$$
\Delta_{\Pi_{new}}=\frac{\partial V_{new}}{\partial S_{new}}=N_{S_{new}}+N_P\cdot\Delta_{P_{new}}=0
$$
可得此时持有的股票数为：
$$
N_{S_{new}}=-N_P\cdot\Delta_{P_{new}} \approx -586.15
$$
故需要新购入股票的数量为：
$$
N_{S_{new}}-N_{S_{dn}}=324.70
$$
因此，重新调整后的现金余额为:
$$
Cash_{new_{dn}}=Cash_{new}-(N_{S_{new}}-N_{S_{dn}}) \cdot S_{new} \approx \$30551.72
$$
具体计算代码如下：


In [15]:
# (1e)

# New delta-neutral stock position
N_S_new_dn = -N_P * delta_P_new

# Stock adjustment
stock_adjust_e = N_S_new_dn - N_S_dn

# Update cash (cash grows first, then adjust stock)
cash_before_rebal = cash_new_d
cash_after_rebal = cash_before_rebal - stock_adjust_e * 24

print(f"(e) New Stock Position: {N_S_new_dn:.2f} shares")
print(f"    Stock Adjustment: +{stock_adjust_e:.2f} shares")
print(f"    New Cash Balance: ${cash_after_rebal:.2f}")

(e) New Stock Position: -586.15 shares
    Stock Adjustment: +324.70 shares
    New Cash Balance: $30551.72
